# My SmartQ Model Diagnostics & Statistical Understanding

I use this notebook to go beyond MAE and RMSE.

I do not want my ML work to become a black box that I cannot explain.

I ask:

- Is each model a good fit?
- Is there evidence of overfitting?
- What does R² tell me?
- Which Linear Regression terms are statistically significant?
- Are some predictors repeating the same information?
- Do the main Linear Regression assumptions hold?
- Which variables matter most to XGBoost?
- What do permutation importance and SHAP tell me?
- Where does the selected model still fail?

My most important lesson here is:

> Statistical significance is not the same as practical importance or predictive usefulness.


## 1. I load the same SmartQ data and recreate the chronological split

I reuse the exact modelling population and feature set from my official experiment.

I do this because diagnostics should explain the same models I evaluated earlier, not a different experiment.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import jarque_bera, durbin_watson
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from xgboost import XGBRegressor, DMatrix

DATASET_NAME = "SmartQ_Synthetic_Operational_Dataset_100k.csv"
candidate_paths = [Path("data") / DATASET_NAME, Path("..") / "data" / DATASET_NAME]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("SmartQ dataset not found.")

df = pd.read_csv(data_path, parse_dates=["scenario_date"])
completed = df[df["status"] == "COMPLETED"].copy()

TARGET = "actual_wait_minutes"
numeric_features = [
    "arrival_offset_minutes","people_ahead","general_waiting","priority_waiting",
    "serving_count","open_general_counters","open_priority_counters",
    "effective_open_counters","counter_utilisation","queue_pressure_index",
    "workload_minutes_ahead","recent_avg_service_minutes_10",
    "recent_avg_wait_minutes_10","recent_throughput_60m",
    "service_target_minutes","hour_of_day",
]
categorical_features = [
    "branch_code","service_code","booking_source",
    "queue_type","day_of_week","is_peak_period",
]
features = numeric_features + categorical_features

dates = sorted(completed["scenario_date"].dt.normalize().unique())
n = len(dates)
train_end = pd.Timestamp(dates[int(n * 0.70) - 1])
val_start = pd.Timestamp(dates[int(n * 0.70)])
val_end = pd.Timestamp(dates[int(n * 0.85) - 1])
test_start = pd.Timestamp(dates[int(n * 0.85)])

train = completed[completed["scenario_date"] <= train_end].copy()
validation = completed[
    (completed["scenario_date"] >= val_start)
    & (completed["scenario_date"] <= val_end)
].copy()
test = completed[completed["scenario_date"] >= test_start].copy()

print(len(train), len(validation), len(test))


## 2. I fit the same preprocessing and official models

I repeat the official model settings so my diagnostics are performed on the same experiment.

I want the diagnostic conclusions to match the models I actually compared.


In [ ]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

X_train, y_train = train[features], train[TARGET].to_numpy()
X_val, y_val = validation[features], validation[TARGET].to_numpy()
X_test, y_test = test[features], test[TARGET].to_numpy()

Xtr = preprocessor.fit_transform(X_train)
Xv = preprocessor.transform(X_val)
Xt = preprocessor.transform(X_test)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=150, max_depth=14, min_samples_leaf=2,
        max_features=1.0, random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0,
        random_state=42, n_jobs=-1, objective="reg:squarederror",
        tree_method="hist"
    ),
}

for model in models.values():
    model.fit(Xtr, y_train)


## 3. I check good fit and generalisation using MAE, RMSE and R²

**R² (R-squared)** tells me how much of the variation in waiting time the model explains.

A high training R² alone is not enough.

I also want validation and test performance to stay strong, because that tells me whether the model generalises beyond the data it learned from.


In [ ]:
def score(y_true, raw_pred):
    pred = np.clip(np.asarray(raw_pred), 0, None)
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": mean_squared_error(y_true, pred) ** 0.5,
        "R2": r2_score(y_true, pred),
    }

rows = []
for name, model in models.items():
    for split_name, X, y in [
        ("train", Xtr, y_train),
        ("validation", Xv, y_val),
        ("test", Xt, y_test),
    ]:
        rows.append({"model": name, "split": split_name, **score(y, model.predict(X))})

fit_table = pd.DataFrame(rows)
display(fit_table)


In [ ]:
pivot = fit_table.pivot(index="model", columns="split", values="MAE")[["train","validation","test"]]
ax = pivot.plot(kind="bar", figsize=(9,5))
ax.set_title("MAE by model and data split")
ax.set_ylabel("MAE (minutes)")
ax.set_xlabel("Model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### What I learn from the fit comparison

Random Forest fits my training data most aggressively because its training MAE is much lower than its validation MAE.

XGBoost has a smaller train-to-validation gap, so its behaviour looks more stable in this experiment.

I do not see catastrophic overfitting because validation and test performance remain close.


## 4. I test statistical significance for Linear Regression

Classic p-values make the most sense for a statistical Linear Regression model.

For inference, I fit a companion OLS model using reference-category encoding and **HC3 robust standard errors**.

I use HC3 because I found that the residual variance is not constant across queue conditions.


In [ ]:
train_ols = train.copy()
for col in numeric_features:
    if train_ols[col].isna().any():
        train_ols[col] = train_ols[col].fillna(train_ols[col].median())

formula = TARGET + " ~ " + " + ".join(
    numeric_features + [f"C({c})" for c in categorical_features]
)

ols = smf.ols(formula, data=train_ols).fit(cov_type="HC3")

coef_table = pd.DataFrame({
    "coefficient": ols.params,
    "std_error_HC3": ols.bse,
    "p_value": ols.pvalues,
    "ci_low": ols.conf_int()[0],
    "ci_high": ols.conf_int()[1],
})
coef_table["significant_0_05"] = coef_table["p_value"] < 0.05

print(f"R²: {ols.rsquared:.4f}")
print(f"Adjusted R²: {ols.rsquared_adj:.4f}")
display(coef_table.sort_values("p_value", ascending=False))


### How I understand a p-value

A p-value asks, roughly:

> If there were really no coefficient effect, how surprising would my result be?

A common threshold is p < 0.05.

Because I have more than 64,000 training rows, even tiny effects can become statistically significant.

So I do **not** automatically treat a small p-value as proof that a feature is important for prediction.


## 5. I compare group significance and effect sizes

I compare a few simple EDA groups.

I use **Cohen's d** because it helps me separate:

- statistically detectable differences;
- differences that are actually large in standardised terms.

I do this because a huge dataset can make tiny differences look statistically impressive.


In [ ]:
def welch_and_d(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    t, p = stats.ttest_ind(a, b, equal_var=False)
    pooled_sd = np.sqrt(
        ((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1))
        / (len(a)+len(b)-2)
    )
    d = (a.mean() - b.mean()) / pooled_sd
    return t, p, d

comparisons = []
for label, column, a_value, b_value in [
    ("General vs Priority", "queue_type", "GENERAL", "PRIORITY"),
    ("Appointment vs Walk-in", "booking_source", "APPOINTMENT", "WALK_IN"),
    ("Peak vs Non-peak", "is_peak_period", True, False),
]:
    a = completed.loc[completed[column] == a_value, TARGET]
    b = completed.loc[completed[column] == b_value, TARGET]
    t, p, d = welch_and_d(a, b)
    comparisons.append({
        "comparison": label,
        "mean_a": a.mean(),
        "mean_b": b.mean(),
        "p_value": p,
        "cohens_d": d,
    })

display(pd.DataFrame(comparisons))

service_groups = [
    group[TARGET].to_numpy()
    for _, group in completed.groupby("service_name")
]
f_stat, p_value = stats.f_oneway(*service_groups)

grand_mean = completed[TARGET].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in service_groups)
ss_total = ((completed[TARGET] - grand_mean)**2).sum()
eta_squared = ss_between / ss_total

print("Service ANOVA F:", f_stat)
print("Service ANOVA p:", p_value)
print("Service eta-squared:", eta_squared)


## 6. I check multicollinearity using VIF

**Multicollinearity** means several predictors repeat similar information.

**VIF (Variance Inflation Factor)** helps me measure that overlap.

I care about high VIF mainly when I interpret individual Linear Regression coefficients.


In [ ]:
numeric_train = train[numeric_features].copy()
numeric_train = numeric_train.fillna(numeric_train.median(numeric_only=True))

vif_input = sm.add_constant(numeric_train.astype(float), has_constant="add")
vif_rows = []
for i, column in enumerate(vif_input.columns):
    if column == "const":
        continue
    vif_rows.append({
        "feature": column,
        "VIF": variance_inflation_factor(vif_input.values, i),
    })

vif_table = pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)
display(vif_table)

top = vif_table.head(10).sort_values("VIF")
ax = top.plot(kind="barh", x="feature", y="VIF", legend=False, figsize=(9,5))
ax.axvline(10, linestyle="--")
ax.set_title("VIF: strongest overlapping numeric predictors")
ax.set_xlabel("VIF")
plt.tight_layout()
plt.show()


### What I found in SmartQ

Queue pressure, people ahead, serving count, counter utilisation, general waiting and workload ahead have high VIF.

That makes sense because they describe related parts of the same queue state.

I keep them in my tree models because prediction is my main goal.

I do **not** treat every Linear Regression coefficient as a clean independent causal effect.


## 7. I test Linear Regression assumptions

I use formal tests to understand whether the classical OLS assumptions fit my residual behaviour.

I do this to learn where standard statistical interpretation needs extra caution.


In [ ]:
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(ols.resid, ols.model.exog)
jb_stat, jb_p, jb_skew, jb_kurtosis = jarque_bera(ols.resid)
dw = durbin_watson(ols.resid)

assumptions = pd.DataFrame([
    ["Breusch-Pagan LM", bp_lm, bp_lm_p],
    ["Breusch-Pagan F", bp_f, bp_f_p],
    ["Jarque-Bera", jb_stat, jb_p],
    ["Durbin-Watson", dw, np.nan],
], columns=["diagnostic","statistic","p_value"])

display(assumptions)
print("Residual skewness:", jb_skew)
print("Residual kurtosis:", jb_kurtosis)


The Breusch-Pagan result shows **heteroscedasticity**, which means my error size changes across queue conditions.

The Jarque-Bera result shows my residuals are not normally distributed and have heavy tails.

Because of that, I use robust HC3 standard errors rather than pretending the classical assumptions are perfect.


## 8. I inspect residuals and negative raw predictions

A **residual** is:

actual wait - predicted wait

If the residual is positive, my model predicted too low.

If the residual is negative, my model predicted too high.

I use residuals to understand the shape and bias of model errors instead of only looking at one average score.


In [ ]:
residual_rows = []
for name, model in models.items():
    for split_name, X, y in [
        ("train", Xtr, y_train),
        ("validation", Xv, y_val),
        ("test", Xt, y_test),
    ]:
        raw = model.predict(X)
        pred = np.clip(raw, 0, None)
        residual = y - pred
        residual_rows.append({
            "model": name,
            "split": split_name,
            "mean_residual": residual.mean(),
            "median_residual": np.median(residual),
            "negative_raw_predictions": int((raw < 0).sum()),
        })

display(pd.DataFrame(residual_rows))

xgb_test_raw = models["XGBoost"].predict(Xt)
xgb_test_pred = np.clip(xgb_test_raw, 0, None)
xgb_residual = y_test - xgb_test_pred

fig, ax = plt.subplots(figsize=(8,5))
ax.hist(xgb_residual, bins=70)
ax.set_title("XGBoost test residual distribution")
ax.set_xlabel("Residual = actual - predicted (minutes)")
ax.set_ylabel("Customers")
plt.show()


My XGBoost model produces some small negative raw predictions.

Waiting time cannot be negative in the real world, so my customer-facing prediction helper applies:

`max(0, prediction)`

I keep this rule explicit because it is a real-world constraint, not something I want hidden inside the model.


## 9. I calculate permutation importance

Permutation importance asks:

> If I destroy one feature's information by shuffling it, how much worse does my model become?

A large increase in MAE tells me the feature carried useful predictive information.


In [ ]:
xgb_model = models["XGBoost"]
baseline_mae = mean_absolute_error(y_test, xgb_test_pred)

rng = np.random.default_rng(42)
permutation_rows = []
X_test_raw = test[features].copy()

for feature in features:
    shuffled = X_test_raw.copy()
    values = shuffled[feature].to_numpy(copy=True)
    rng.shuffle(values)
    shuffled[feature] = values

    pred = np.clip(
        xgb_model.predict(preprocessor.transform(shuffled)),
        0,
        None,
    )
    mae = mean_absolute_error(y_test, pred)
    permutation_rows.append({
        "feature": feature,
        "permuted_mae": mae,
        "mae_increase": mae - baseline_mae,
    })

permutation_table = pd.DataFrame(permutation_rows).sort_values(
    "mae_increase", ascending=False
)
display(permutation_table)

top = permutation_table.head(10).sort_values("mae_increase")
ax = top.plot(
    kind="barh", x="feature", y="mae_increase",
    legend=False, figsize=(9,5)
)
ax.set_title("Permutation importance")
ax.set_xlabel("Increase in test MAE after shuffling (minutes)")
plt.tight_layout()
plt.show()


## 10. I use SHAP / TreeSHAP

**SHAP** estimates how much each feature pushes a model prediction up or down.

For learning, I use XGBoost's built-in TreeSHAP contributions on a fixed 5,000-row test sample.

I use this to understand model behaviour, not to claim that the features are causal.


In [ ]:
feature_names = preprocessor.get_feature_names_out()
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(Xt), size=min(5000, len(Xt)), replace=False)

dmatrix = DMatrix(Xt[sample_idx], feature_names=list(feature_names))
contributions = xgb_model.get_booster().predict(
    dmatrix, pred_contribs=True
)

shap_table = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(contributions[:, :-1]).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

display(shap_table.head(15))


## 11. My final diagnostic conclusion

I keep XGBoost as my integration candidate.

I do not change the model after diagnostics because:

- validation/test performance remains strong;
- I do not see a catastrophic generalisation collapse;
- the strongest features make operational sense;
- permutation importance and SHAP agree on the main queue drivers;
- the statistical problems mostly affect how I interpret Linear Regression coefficients rather than invalidating the tree-model comparison.

What I learned:

- Random Forest fits training data more aggressively.
- XGBoost generalises more evenly in this experiment.
- several engineered queue variables are highly correlated.
- p-values alone are not enough to judge usefulness.
- classical OLS assumptions are not perfect.
- robust inference is more appropriate.
- workload ahead, people ahead and counter capacity are major predictive signals.
- busy queues remain the hardest operating condition.
- non-negative clipping is necessary for customer-facing regression predictions.

The diagnostics did not give me a new winner.

They gave me a better understanding of the model I already selected.
